[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/slip_systems/slip_systems.ipynb)

# Slip in Metal Crystals

This example computes which slip systems of a metal crystal yield under a pull, and how slipping turns the crystal. A copper wire bent stays bent: inside each crystal of the metal, layers of atoms have slid over one another along a few planes, in a few directions, the crystal's slip systems. A pull shears each system by a different amount, and the one sheared most slips first. Which one that is depends only on the direction of the pull relative to the crystal, and as the crystal slips the pull's direction turns through it, until a second system takes its share.

A slip plane is modelled as a vector and the pull's axis as a direction, an antivector; a slip direction is the line where two slip planes meet, `plane ^ other`, a bivector whose sense the order of the planes sets, and which drops out of every slip, since the line enters it twice. The pull is a stress, a map from a plane to the traction on it, `axis * (Plane & axis)`, and the shear on a system is that map read along its slip direction, `direction.dot(tension(axis)(plane))`. A slip is the shear map `Direction + amount * direction * (plane & Direction)`, and its move of the pull's axis turns the axis by the bivector of the plane they span, `axis.commutator(move)`.

In the index notation of crystal plasticity a slip by $\gamma$ on a system with slip direction $b$ and plane normal $n$ is the displacement gradient $\gamma\, b_i n_j$, and the pull a stress $\sigma\, t_i t_j$ along the unit axis $t$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from examples.animation import save_figure, show_animation
from examples.mechanics.slip_systems import render

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# A slip plane through the origin.
Plane = ga.gatype.vector()
# A direction in the crystal, the pull's axis: an antivector with intrinsic orientation.
Direction = ga.gatype.antivector()
# A slip direction, the line where two slip planes meet: a bivector, its sense set by the order of the planes.
Line = ga.gatype.bivector()
# Traction on a plane.
Stress = ga.gatype((Direction, Plane))                     # Direction <- Plane

## 1. Twelve slip systems

The four densest planes of the crystal have normals along the body diagonals of its cube, and the densest directions within each plane are the lines where it meets the other three: each pair of planes meets in one line, so a plane's slip directions are its meets with the others, `plane ^ other`. Four planes with three directions each make twelve systems, kept as a batch of four by three; each direction lies in two planes, so the six lines where the planes cross carry two systems each.

In [ ]:
# The normals of the four slip planes of a face-centred cubic crystal, the {111} planes.
plane_normals = [[1, 1, 1], [-1, 1, 1], [-1, -1, 1], [1, -1, 1]]

In [ ]:
def slip_directions(planes: Plane) -> Line:
    """The three slip directions of each slip plane: the lines where it meets each of the others, each
    normalized."""
    count = planes.shape[-1]
    others = (np.arange(count)[:, None] + np.arange(1, count)) % count         # [planes, directions]
    return (planes[:, None] ^ planes[others]).normalized()                    # [planes, directions] Line


planes = mv.vector(plane_normals).normalized()                                 # [planes] Plane
slips = slip_directions(planes)                                                # [planes, directions] Line
print("largest part of a slip direction across its plane:", np.abs(render.scalars(planes[:, None] & slips)).max())

## 2. The Schmid factor

A pull along an axis is a stress, a map from a plane to the traction on it: the axis, by the plane's pairing with it, `axis * (Plane & axis)`. Read along a slip direction, the traction on the slip plane is the shear stress that drives the slip, the Schmid factor times the pull. It is largest, a half, when the axis lies at forty-five degrees to both the normal and the direction, and the system with the largest factor yields first. Along a cube axis eight systems tie at one over the root of six; anywhere else one system leads, and over the sphere of pull directions the leaders divide it into regions, each coloured below by its system and lightened where its factor is small.

In the notation of crystal plasticity the Schmid factor reads as $m = \cos\phi\,\cos\lambda$, the cosines of the pull's angles to the plane normal and to the slip direction.

In [ ]:
# The samples across the stereographic disk of pull directions.
n_pixels = 1000

In [ ]:
def tension(axis: Direction) -> Stress:
    """A unit pull along each axis: the traction on a plane is the axis, by the plane's pairing with it."""
    return axis * (Plane & axis)                                              # [...] Direction <- Plane


def schmid(axis: Direction, planes: Plane, directions: Line) -> Scalar:
    """The shear stress each pull resolves along each slip system, per unit pull: the traction on the slip
    plane, along the slip direction."""
    return directions.dot(tension(axis[..., None, None])(planes[:, None]))   # [..., planes, directions] Scalar


# Pull directions sampled on the pixels of the chart: each point of the plane across [001] lifts to the direction
# that projects onto it, [001] reflected in the mirror through [001] plus the point.
across = np.linspace(-1, 1, n_pixels)
offsets = mv.x * across + mv.y * across[:, None]                               # [rows, columns] Vector
directions = (mv.z + offsets).normalized() >> mv.z.dual()                      # [rows, columns] Direction
factors = schmid(directions, planes, slips)                                    # [rows, columns, planes, directions] Scalar
along_cube_axis = np.abs(render.scalars(schmid(mv.z.dual(), planes, slips)))  # [planes, directions]

print("Schmid factors along [001]:\n", along_cube_axis.round(4))

## 3. The crystal's turn under the pull

A small slip on a system shears the crystal: each point moves along the slip direction by its height above the slip plane, the map `Direction + amount * direction * (plane & Direction)`, and slips on several systems add their moves. Every system slips in the sense the pull drives it, at a rate rising steeply with its Schmid factor, as the slip rate of a metal rises with a high power of the shear stress: the leading system takes almost all of the slip, and two systems with equal factors share it alike. The specimen stays lined up with the pull, so in the crystal's own frame it is the pull's axis that moves. The part of its move along it only stretches it; the part across it turns it, by the bivector `axis.commutator(move)`, in the plane of the axis and the slip direction, towards the slip direction.

In [ ]:
# The axes the pulls start along, inside the standard triangle between [001], [011] and [111]; the slip of each
# step, how steeply a system's rate rises with its Schmid factor, and the steps taken; and the frames drawn, and the
# duration of each.
starts = [[0.05, 0.3, 1], [0.15, 0.32, 1], [0.03, 0.5, 1], [0.2, 0.5, 1], [0.05, 0.75, 1],
          [0.3, 0.6, 1], [0.12, 0.62, 1], [0.4, 0.55, 1]]
amount = 0.01
steepness = 30
n_steps = 500
n_frames = 50
duration_ms = 70

In [ ]:
def slipped(axis: Direction, planes: Plane, directions: Line, amounts: Scalar) -> Direction:
    """Each axis after a small slip on every system by its amount: each slip moves the axis along its direction
    by its height above its plane, the moves add, and the part of their sum across the axis turns it, in the
    plane of the axis and the move."""
    # Each slip line enters twice, through its Schmid factor and here, so the sense the order of its planes
    # gave it drops out of the move.
    moves = amounts * directions * (planes[:, None] & axis[..., None, None])  # [..., planes, directions] Direction
    move = moves.sum(axis=-1).sum(axis=-1)                                    # [...] Direction
    # The commutator of two directions is the bivector of the plane they span.
    return (axis.commutator(move) / 2).exp() >> axis                         # [...] Direction


def power_law(factors: Scalar) -> Scalar:
    """How much each system slips in a step: the step's slip, in the sign of the Schmid factor, scaled by the
    factor's size over the leading one raised to the steepness. The one step outside the algebra."""
    values = render.scalars(factors)                                           # [..., planes, directions]
    leading = np.abs(values).max(axis=(-2, -1), keepdims=True)
    return factors * (amount * np.abs(values) ** (steepness - 1) / leading ** steepness)


def rotation(axis: Direction) -> Iterator[tuple[Direction, Scalar]]:
    """Each pull's axis after every slip step, and how much each system slipped in the step."""
    for _ in range(n_steps):
        amounts = power_law(schmid(axis, planes, slips))                       # [starts, planes, directions] Scalar
        axis = slipped(axis, planes, slips, amounts)
        yield axis, amounts


start_axes = mv.vector(starts).normalized().dual()                             # [starts] Direction
moved, amounts = zip(*rotation(start_axes))
paths = stack([start_axes, *moved])                                            # [steps + 1, starts] Direction
amounts = stack(amounts)                                                       # [steps, starts, planes, directions] Scalar
carried = np.sort(np.abs(render.scalars(amounts)).sum(axis=0).reshape(len(starts), -1), axis=-1)   # [starts, systems]
print("share of each pull's slip on its two leading systems:", (carried[:, -2:].sum(axis=-1) / carried.sum(axis=-1)).round(3))
save_figure(render.draw_chart(directions, factors, planes, slips, paths, amounts), "slip_systems")
show_animation(render.animate_paths(directions, factors, paths, amounts, n_frames), "slip_systems_paths", duration_ms)

## Reading the chart

The left disk holds every direction a pull can take, seen down the cube axis [001]: the centre is a pull along [001], the rim pulls across it, and a pull and its opposite are the same, so each is drawn on its upper half. Each colour is one of the twelve slip systems, painting the directions for which it yields first. The black triangles are the four plane normals and the dots the slip directions; together they cut the disk into copies of one triangle, the standard triangle between [001], [011] and [111], which the cube's symmetries map onto one another, so one copy tells the whole story. The fades follow the leading Schmid factor: full colour where it reaches a half, the pull at forty-five degrees to both the plane and the direction, and white where it falls to a quarter. The palest patches sit around the plane normals: a pull along a normal puts no shear on its own plane and little on the other three.

The right panel is the standard triangle close up. Inside it one system, the orange one, leads, slipping on the plane with normal [-111] along [101], so every pull's axis turns towards [101], the diamond at the bottom right, along the great circle through it. It never gets there: it first meets the edge between [001] and [111], where the brown system's factor catches up with the orange one's. From then on the two slip together, each pushing the axis back across the edge into its own region, and their sum runs the axis along the edge to [112], where the two shear it equally and it stops turning. The paths are drawn in the colours of the systems slipping, mixed by how much each slips: orange while one system works alone, a mix of orange and brown along the edge. Every start ends at [112], whichever way it set out. The rate rises steeply but continuously with the factor, so the two systems share each step in proportion and the axis runs straight along the edge rather than zigzagging across it.

In [ ]:
# checks
# Every slip direction lies in its plane; no Schmid factor exceeds a half, and along [001] eight systems share
# the largest, one over the root of six.
assert np.abs(render.scalars(planes[:, None] & slips)).max() < 1e-12
assert np.abs(render.scalars(factors)).max() <= 0.5 + 1e-12
assert np.isclose(along_cube_axis.max(), 1 / np.sqrt(6)) and np.isclose(along_cube_axis, along_cube_axis.max()).sum() == 8
# Every pull's axis ends at [112], two systems carrying nearly all of its slip on the way.
ends = render.coordinates(paths[-1])                                           # [starts, 3]
assert (np.sort(ends, axis=-1) @ (np.array([1, 1, 2]) / np.sqrt(6)) > 0.999).all()
assert (carried[:, -2:].sum(axis=-1) / carried.sum(axis=-1) > 0.95).all()